# Phase 5 -- Layer-Specific SMI Effects, Random-Subsample Trial-Matched (DREADD saline/DCZ cohort)

Mirrors `5.LayerSpecific_TrialMatched.ipynb` (which itself mirrors `5.LayerSpecific.py`'s core layer/depth-group analysis) -- but consumes `4.SessionComparison_TrialMatched_RandomSubsample.ipynb`'s per-cell tables (`{group}_random_subsampled_comparison_table.csv`) instead of the sequential-block notebook's. So `condition` here has just **two** values per group: `'saline'` and `'dcz_random_subsampled'` (the aggregate across `N_REPEATS` random draws), not a `dcz_block_1..n` sequence.

**Every layer/depth-group function below is identical to `5.LayerSpecific_TrialMatched.ipynb`'s** -- they're already generic over whatever's in `condition`. The only thing that changes between the two Phase-5 notebooks is which directory/filename pattern gets loaded, and `_condition_order_and_colors` (shared design, not hardcoded to either method's naming: sorts `dcz_block_N`-style names numerically, anything else -- like `dcz_random_subsampled` -- alphabetically, and colors every non-saline condition from the same Purples gradient regardless of naming).

**Pseudo-replication caveat -- different (milder) shape here than the sequential-block notebook's.** `5.LayerSpecific.py`'s own docstring flags that its pooled-cell tests treat cells within one group as independent, when there's really only one saline session and one paired dcz session per group. The sequential-block notebook's version of this caveat is *worse* than that (blocks share overlapping trials with each other). **This one is arguably back to the *original*, milder version of the caveat**: `dcz_random_subsampled`'s SMI/valid values are already an aggregate across `N_REPEATS` independent random draws (not one single trial subset), so it's a more stable per-cell estimate than any single block or single session was -- but it's still cells-within-one-session-pair pseudo-replication for the purposes of a per-layer Kruskal-Wallis/interaction test, same as `5.LayerSpecific.py`'s original 5.1/5.2/5.4. Treat per-group, per-layer p-values here with the same level of caution `5.LayerSpecific.py` itself already recommends (see its Functions 5.11-5.14 for the corrected, paired-across-groups version of this question) -- not the extra caution the sequential-block notebook's layer analysis needs.

In [ ]:
import sys
sys.path.insert(0, r"C:\Users\jasmineyeo\Documents\GitHub\V1_SpatialModulation")

import os
import glob
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Qt5Agg')  # for plt.show() popups
import matplotlib.pyplot as plt
from matplotlib import rcParams
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
from scipy.stats import kruskal, mannwhitneyu, rankdata

rcParams['legend.fontsize'] = 20
rcParams['axes.labelsize'] = 20
rcParams['axes.titlesize'] = 25
rcParams['xtick.labelsize'] = 20
rcParams['ytick.labelsize'] = 20

# Point this at whichever animal you're processing -- kept for both so
# switching doesn't leave other TEST_* constants pointing at the wrong
# animal.
TEST_ANIMAL_DIR_JSY090 = r"D:\V1_SpatialModulation\2p\V1_prism_DREADD\JSY090_V1prism_DREADD"
TEST_ANIMAL_DIR_JSY093 = r"D:\V1_SpatialModulation\2p\V1_prism_DREADD\JSY093_V1prism_DREADD"
TEST_ANIMAL_DIR = TEST_ANIMAL_DIR_JSY090

## Setup -- `load_all_group_dfs_from_random_subsample`

Loads every `{group}_random_subsampled_comparison_table.csv` already saved by `4.SessionComparison_TrialMatched_RandomSubsample.ipynb`'s `save_random_subsample_outputs` -- the one piece that's actually different from `5.LayerSpecific_TrialMatched.ipynb` besides `_condition_order_and_colors`'s docstring.

In [ ]:
def load_all_group_dfs_from_random_subsample(output_dir):
    """
    Load every group's per-cell table already saved by
    4.SessionComparison_TrialMatched_RandomSubsample.ipynb's
    save_random_subsample_outputs
    ('{group_name}_random_subsampled_comparison_table.csv').

    Parameters
    ----------
    output_dir : str
        e.g. os.path.join(ANIMAL_DIR, 'Phase4_TrialMatched_RandomSubsample_Results').

    Returns
    -------
    all_group_dfs : dict
        {group_name: df}.
    """
    csv_paths = sorted(glob.glob(os.path.join(output_dir, '*_random_subsampled_comparison_table.csv')))

    if not csv_paths:
        raise FileNotFoundError(f"No *_random_subsampled_comparison_table.csv found in {output_dir} -- "
                                 "has 4.SessionComparison_TrialMatched_RandomSubsample.ipynb's save step "
                                 "been run for this animal?")

    all_group_dfs = {}
    for csv_path in csv_paths:
        group_name = os.path.basename(csv_path)[:-len('_random_subsampled_comparison_table.csv')]
        df = pd.read_csv(csv_path)
        all_group_dfs[group_name] = df
        print(f"Loaded '{group_name}': {len(df)} cell-rows <- {csv_path}")

    print(f"\nLoaded {len(all_group_dfs)} group(s) from {output_dir}: {list(all_group_dfs.keys())}")
    return all_group_dfs

## Function -- `compare_smi_across_conditions`

Reimplemented unchanged from `5.LayerSpecific.py`'s Function 5.1 / `5.LayerSpecific_TrialMatched.ipynb`'s own copy.

In [ ]:
def compare_smi_across_conditions(df, group_col='condition', value_col='SMI', filter_col='valid'):
    """
    Kruskal-Wallis omnibus + pairwise Mann-Whitney U (Holm-corrected)
    across whatever categories are present.

    Parameters
    ----------
    df : pandas.DataFrame
        One group's or one layer's/depth-group's table.
    group_col, value_col, filter_col : str

    Returns
    -------
    result : dict or None
        None (with a printed message) if fewer than 2 categories remain
        after filtering. Otherwise:
        {'group_medians': {cat: median}, 'group_n': {cat: n},
         'omnibus_stat', 'omnibus_p',
         'pairwise': DataFrame(cond_a, cond_b, median_diff, U_stat, p_raw, p_holm)}.
    """
    filtered = df[df[filter_col]]
    categories = [c for c in filtered[group_col].unique() if pd.notna(c)]

    if len(categories) < 2:
        print(f"Only {len(categories)} category(ies) present after filtering on '{filter_col}' "
              f"-- nothing to compare ({categories}).")
        return None

    samples = {cat: filtered.loc[filtered[group_col] == cat, value_col].to_numpy()
               for cat in categories}

    group_medians = {cat: float(np.median(vals)) for cat, vals in samples.items()}
    group_n = {cat: len(vals) for cat, vals in samples.items()}

    omnibus_stat, omnibus_p = kruskal(*samples.values())

    pairwise_rows = []
    for cat_a, cat_b in combinations(categories, 2):
        u_stat, p_raw = mannwhitneyu(samples[cat_a], samples[cat_b], alternative='two-sided')
        pairwise_rows.append({
            'cond_a': cat_a, 'cond_b': cat_b,
            'median_diff': group_medians[cat_a] - group_medians[cat_b],
            'U_stat': u_stat, 'p_raw': p_raw,
        })

    pairwise_df = pd.DataFrame(pairwise_rows)
    if len(pairwise_df) > 0:
        _, p_holm, _, _ = multipletests(pairwise_df['p_raw'], method='holm')
        pairwise_df['p_holm'] = p_holm

    print(f"Categories ({group_col}): {categories}")
    print(f"  n per category: {group_n}")
    print(f"  median {value_col} per category: {group_medians}")
    print(f"  Kruskal-Wallis: H={omnibus_stat:.3f}, p={omnibus_p:.4f}")
    print(f"\n  Pairwise (Holm-corrected):")
    print(pairwise_df.to_string(index=False))

    return {
        'group_medians': group_medians,
        'group_n': group_n,
        'omnibus_stat': omnibus_stat,
        'omnibus_p': omnibus_p,
        'pairwise': pairwise_df,
    }

## Function -- `compare_smi_by_layer`

Reimplemented unchanged -- loops `compare_smi_across_conditions` over each layer present (canonical L2/3 -> L4 -> L5 -> L6 order).

In [ ]:
CANONICAL_LAYER_ORDER = ['L2/3', 'L4', 'L5', 'L6']


def _layer_order(layers_present):
    return ([l for l in CANONICAL_LAYER_ORDER if l in layers_present]
            + [l for l in layers_present if l not in CANONICAL_LAYER_ORDER])


def compare_smi_by_layer(df, layer_col='layer', group_col='condition', value_col='SMI', filter_col='valid'):
    """
    Loop compare_smi_across_conditions over each layer present.

    Parameters
    ----------
    df : pandas.DataFrame
        One group's table.
    layer_col, group_col, value_col, filter_col : str

    Returns
    -------
    results : dict
        {layer: compare_smi_across_conditions(...) result or None}.
    summary_df : pandas.DataFrame
        One row per (layer, pairwise comparison) that had 2+ categories.
    """
    layers_present = [l for l in df[layer_col].dropna().unique()]
    layer_order = _layer_order(layers_present)

    results = {}
    summary_rows = []
    for layer in layer_order:
        print(f"\n--- Layer {layer} ---")
        layer_df = df[df[layer_col] == layer]
        result = compare_smi_across_conditions(layer_df, group_col=group_col, value_col=value_col,
                                                filter_col=filter_col)
        results[layer] = result
        if result is not None:
            for _, row in result['pairwise'].iterrows():
                summary_rows.append({
                    'layer': layer, 'cond_a': row['cond_a'], 'cond_b': row['cond_b'],
                    'median_diff': row['median_diff'], 'p_holm': row['p_holm'],
                })

    summary_df = pd.DataFrame(summary_rows)
    if len(summary_df) > 0:
        print("\n=== Summary across layers (each layer's own Holm correction -- not corrected across layers) ===")
        print(summary_df.to_string(index=False))

    return results, summary_df

## Function -- `_condition_order_and_colors`

Same generic version `5.LayerSpecific_TrialMatched.ipynb` now uses (retroactively fixed there too) -- saline first, then every other condition sorted numerically if its name ends in `_<number>`, alphabetically otherwise, each colored from a Purples gradient. Not hardcoded to either method's condition-naming scheme.

In [ ]:
def _condition_order_and_colors(conditions_present):
    """
    Order + color scheme for whatever's in the condition column: saline
    first (if present), then every other condition -- sorted numerically
    if its name ends in '_<number>' (e.g. dcz_block_1, dcz_block_2, ...),
    alphabetically otherwise (e.g. dcz_random_subsampled) -- each given
    its own shade from a Purples gradient.

    Parameters
    ----------
    conditions_present : iterable of str

    Returns
    -------
    order : list of str
    color_by_category : dict
    """
    conditions_present = list(conditions_present)

    def _sort_key(cond):
        cond_str = str(cond)
        if '_' in cond_str:
            suffix = cond_str.rsplit('_', 1)[1]
            if suffix.isdigit():
                return (0, int(suffix), cond_str)
        return (1, 0, cond_str)

    order = ['saline'] if 'saline' in conditions_present else []
    others = sorted([c for c in conditions_present if c != 'saline'], key=_sort_key)
    order += others

    color_by_category = {'saline': 'tab:orange'}
    n_others = max(len(others), 1)
    other_colors = plt.cm.Purples(np.linspace(0.4, 0.9, n_others))
    for i, cond in enumerate(others):
        color_by_category[cond] = other_colors[i]

    return order, color_by_category


def _display_condition_label(cond):
    """
    Short display label for a condition's x-axis tick text. Only
    'dcz_random_subsampled' needs this -- one long word wraps onto two
    shorter lines ('dcz' / 'subsampled') so panels stay compact instead
    of needing a wide rotated single-line label.
    """
    if cond == 'dcz_random_subsampled':
        return 'dcz\nsubsampled'
    return str(cond)


def _p_to_stars(p):
    """Convert a p-value to a significance-star string ('ns' if >= .05)."""
    if p < 0.001:
        return '***'
    elif p < 0.01:
        return '**'
    elif p < 0.05:
        return '*'
    return 'ns'


def _annotate_pairwise_significance_brackets(ax, category_order, data_by_category, pairwise_df):
    """
    Draw a significance bracket (stars from Holm-corrected p, 'ns' if not
    significant) for every row of a compare_smi_across_conditions-style
    pairwise_df whose both categories are present on this panel. With
    only two conditions in this notebook (saline, dcz_random_subsampled)
    that's normally just one bracket per panel, but this is written
    generically over however many categories/pairs are actually present.

    Parameters
    ----------
    ax : matplotlib.axes.Axes
    category_order : list of str
        x-axis category order (1-indexed positions in the violinplot).
    data_by_category : list of np.ndarray
        Same order as category_order, used to find headroom above the data.
    pairwise_df : pandas.DataFrame or None
        compare_smi_across_conditions result's 'pairwise' table (needs
        'cond_a', 'cond_b', 'p_holm' columns).

    Returns
    -------
    y_top : float
        Highest y-coordinate used by a bracket (or the data max if none
        were drawn), so the caller can extend ylim to fit.
    """
    data_max = max((v.max() for v in data_by_category if len(v)), default=1.0)

    if pairwise_df is None or len(pairwise_df) == 0:
        return data_max

    y0 = max(data_max, 1.0) + 0.15
    step = 0.13
    tick = 0.03

    y_top = data_max
    i = 0
    for _, row in pairwise_df.iterrows():
        if row['cond_a'] not in category_order or row['cond_b'] not in category_order:
            continue
        x_a = category_order.index(row['cond_a']) + 1
        x_b = category_order.index(row['cond_b']) + 1
        y = y0 + step * i
        i += 1
        ax.plot([x_a, x_a, x_b, x_b], [y - tick, y, y, y - tick], color='black', lw=1)
        ax.text((x_a + x_b) / 2, y + 0.01, _p_to_stars(row['p_holm']),
                ha='center', va='bottom', fontsize=11)
        y_top = max(y_top, y)

    return y_top

## Function -- `plot_smi_by_layer`

Grid of violin+strip plots, one panel per layer present.

In [ ]:
def plot_smi_by_layer(df, layer_col='layer', group_col='condition', value_col='SMI', filter_col='valid', title='',
                      stats_by_layer=None):
    """
    Grid of violin+strip plots, one panel per layer present.

    Parameters
    ----------
    df : pandas.DataFrame
    layer_col, group_col, value_col, filter_col : str
    title : str
    stats_by_layer : dict, optional
        {layer: compare_smi_across_conditions(...) result or None} --
        i.e. compare_smi_by_layer's first return value. If given, draws a
        significance bracket (stars from Holm-corrected p, 'ns' if not
        significant) between saline and dcz_random_subsampled on each
        panel.

    Returns
    -------
    fig : matplotlib.figure.Figure
    """
    filtered = df[df[filter_col]]
    layers_present = [l for l in filtered[layer_col].dropna().unique()]
    layer_order = _layer_order(layers_present)

    n_layers = len(layer_order)
    n_cols = 2
    n_rows = int(np.ceil(n_layers / n_cols)) if n_layers > 0 else 1
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(8 * n_cols, 7 * n_rows))
    axes = np.atleast_1d(axes).flatten()

    for ax, layer in zip(axes, layer_order):
        layer_df = filtered[filtered[layer_col] == layer]
        category_order, color_by_category = _condition_order_and_colors(layer_df[group_col].unique())

        data_by_category = [layer_df.loc[layer_df[group_col] == cat, value_col].to_numpy()
                            for cat in category_order]

        if len(data_by_category) == 0 or all(len(d) == 0 for d in data_by_category):
            ax.set_title(f"{layer} (no data)")
            ax.axis('off')
            continue

        parts = ax.violinplot(data_by_category, showmedians=True)
        for i, body in enumerate(parts['bodies']):
            body.set_facecolor(color_by_category.get(category_order[i], 'gray'))
            body.set_alpha(0.4)

        rng = np.random.default_rng(0)
        for i, vals in enumerate(data_by_category):
            jitter = rng.uniform(-0.08, 0.08, size=len(vals))
            ax.scatter(np.full(len(vals), i + 1) + jitter, vals,
                       color=color_by_category.get(category_order[i], 'gray'), s=12, alpha=0.5)

        layer_stats = stats_by_layer.get(layer) if stats_by_layer else None
        pairwise_df = layer_stats['pairwise'] if layer_stats is not None else None
        y_top = _annotate_pairwise_significance_brackets(ax, category_order, data_by_category, pairwise_df)

        ax.set_ylim(-1.3, max(1.3, y_top + 0.25))
        ax.set_yticks(np.arange(-1, 1.1, 0.5))
        ax.set_xticks(range(1, len(category_order) + 1))
        ax.set_xticklabels([_display_condition_label(c) for c in category_order], rotation=0, ha='center')
        ax.set_ylabel(value_col)
        ax.set_title(layer)
        ax.axhline(0, color='gray', linestyle='--', alpha=0.5)

    for ax in axes[len(layer_order):]:
        ax.axis('off')

    fig.suptitle(title, fontsize=20, fontweight='bold')
    plt.tight_layout()
    return fig

## Function -- `test_layer_depth_interaction`

Reimplemented unchanged -- `rank(SMI) ~ C(condition) * C(depth_group)`. With only 2 conditions here (saline, dcz_random_subsampled), this is a much smaller/simpler model than the sequential-block notebook's version (which has as many condition-levels as blocks).

In [ ]:
def test_layer_depth_interaction(df, layer_col='layer', group_col='condition', value_col='SMI', filter_col='valid',
                                  deep_layers=('L5', 'L6'), superficial_layers=('L2/3', 'L4')):
    """
    rank(SMI) ~ C(condition) * C(depth_group).

    Parameters
    ----------
    df : pandas.DataFrame
        One group's table.
    layer_col, group_col, value_col, filter_col : str
    deep_layers, superficial_layers : tuple of str

    Returns
    -------
    model_result : statsmodels regression results object, or None
        (fewer than 2 conditions or depth groups present).
    """
    filtered = df[df[filter_col]].copy()

    def _depth(layer):
        if layer in deep_layers:
            return 'deep'
        elif layer in superficial_layers:
            return 'superficial'
        return None

    filtered['depth_group'] = filtered[layer_col].map(_depth)
    filtered = filtered.dropna(subset=['depth_group', group_col, value_col])

    conditions_present = filtered[group_col].unique()
    depths_present = filtered['depth_group'].unique()

    if len(conditions_present) < 2 or len(depths_present) < 2:
        print(f"Not enough categories to test an interaction (conditions={list(conditions_present)}, "
              f"depths={list(depths_present)}) -- skipping.")
        return None

    filtered['SMI_rank'] = rankdata(filtered[value_col])

    formula = f"SMI_rank ~ C({group_col}) * C(depth_group)"
    model_result = smf.ols(formula, data=filtered).fit()

    print(f"\n=== Layer-depth interaction: rank({value_col}) ~ {group_col} * depth_group ===")
    print(model_result.summary().tables[1])

    interaction_terms = [p for p in model_result.params.index if ':' in p]
    if interaction_terms:
        print(f"\nInteraction term(s): {interaction_terms}")
        print("(a significant interaction term means the condition effect's SIZE differs "
              "between deep and superficial layers)")

    return model_result

## Function -- `compare_smi_by_depth_group` / `plot_smi_by_depth_group`

Reimplemented unchanged -- pools layers into `deep` (L5+L6) and `superficial` (L2/3+L4) buckets before comparing.

In [ ]:
def _assign_depth_group(layer_series, deep_layers=('L5', 'L6'), superficial_layers=('L2/3', 'L4')):
    """Map a 'layer' column to 'deep'/'superficial'/None."""
    def _depth(layer):
        if layer in deep_layers:
            return 'deep'
        elif layer in superficial_layers:
            return 'superficial'
        return None
    return layer_series.map(_depth)


def compare_smi_by_depth_group(df, layer_col='layer', group_col='condition', value_col='SMI', filter_col='valid',
                                deep_layers=('L5', 'L6'), superficial_layers=('L2/3', 'L4')):
    """
    Loop compare_smi_across_conditions over the two pooled depth groups
    ('deep' = L5+L6, 'superficial' = L2/3+L4) instead of all four layers.

    Parameters
    ----------
    df : pandas.DataFrame
        One group's table.
    layer_col, group_col, value_col, filter_col : str
    deep_layers, superficial_layers : tuple of str

    Returns
    -------
    results : dict
        {'deep': compare_smi_across_conditions(...) result or None,
         'superficial': ... }.
    summary_df : pandas.DataFrame
        One row per (depth_group, pairwise comparison) that had 2+ categories.
    """
    df = df.copy()
    df['depth_group'] = _assign_depth_group(df[layer_col], deep_layers, superficial_layers)

    results = {}
    summary_rows = []
    for depth_group, layers in (('deep', deep_layers), ('superficial', superficial_layers)):
        print(f"\n--- Depth group: {depth_group} ({'+'.join(layers)}) ---")
        depth_df = df[df['depth_group'] == depth_group]
        result = compare_smi_across_conditions(depth_df, group_col=group_col, value_col=value_col,
                                                filter_col=filter_col)
        results[depth_group] = result
        if result is not None:
            for _, row in result['pairwise'].iterrows():
                summary_rows.append({
                    'depth_group': depth_group, 'cond_a': row['cond_a'], 'cond_b': row['cond_b'],
                    'median_diff': row['median_diff'], 'p_holm': row['p_holm'],
                })

    summary_df = pd.DataFrame(summary_rows)
    if len(summary_df) > 0:
        print("\n=== Summary: deep vs superficial (each depth group's own Holm correction) ===")
        print(summary_df.to_string(index=False))

    return results, summary_df


def plot_smi_by_depth_group(df, layer_col='layer', group_col='condition', value_col='SMI', filter_col='valid',
                             deep_layers=('L5', 'L6'), superficial_layers=('L2/3', 'L4'), title='',
                             stats_by_depth=None):
    """
    Violin+strip plots, one panel for 'deep' and one for 'superficial'.

    stats_by_depth : dict, optional
        {'deep': ..., 'superficial': ...} compare_smi_across_conditions
        results -- i.e. compare_smi_by_depth_group's first return value.
        If given, draws a significance bracket (stars from Holm-corrected
        p, 'ns' if not significant) between saline and
        dcz_random_subsampled on each panel.
    """
    filtered = df[df[filter_col]].copy()
    filtered['depth_group'] = _assign_depth_group(filtered[layer_col], deep_layers, superficial_layers)

    fig, axes = plt.subplots(1, 2, figsize=(16, 7))

    for ax, depth_group in zip(axes, ('deep', 'superficial')):
        depth_df = filtered[filtered['depth_group'] == depth_group]
        category_order, color_by_category = _condition_order_and_colors(depth_df[group_col].unique())

        data_by_category = [depth_df.loc[depth_df[group_col] == cat, value_col].to_numpy()
                            for cat in category_order]

        if len(data_by_category) == 0 or all(len(d) == 0 for d in data_by_category):
            ax.set_title(f"{depth_group} (no data)")
            ax.axis('off')
            continue

        parts = ax.violinplot(data_by_category, showmedians=True)
        for i, body in enumerate(parts['bodies']):
            body.set_facecolor(color_by_category.get(category_order[i], 'gray'))
            body.set_alpha(0.4)

        rng = np.random.default_rng(0)
        for i, vals in enumerate(data_by_category):
            jitter = rng.uniform(-0.08, 0.08, size=len(vals))
            ax.scatter(np.full(len(vals), i + 1) + jitter, vals,
                       color=color_by_category.get(category_order[i], 'gray'), s=12, alpha=0.5)

        depth_stats = stats_by_depth.get(depth_group) if stats_by_depth else None
        pairwise_df = depth_stats['pairwise'] if depth_stats is not None else None
        y_top = _annotate_pairwise_significance_brackets(ax, category_order, data_by_category, pairwise_df)

        ax.set_ylim(-1.3, max(1.3, y_top + 0.25))
        ax.set_yticks(np.arange(-1, 1.1, 0.5))
        ax.set_xticks(range(1, len(category_order) + 1))
        ax.set_xticklabels([_display_condition_label(c) for c in category_order], rotation=0, ha='center')
        ax.set_ylabel(value_col)
        ax.set_title(depth_group)
        ax.axhline(0, color='gray', linestyle='--', alpha=0.5)

    fig.suptitle(title, fontsize=20, fontweight='bold')
    plt.tight_layout()
    return fig

## Function -- `summarize_layer_sample_sizes`

Reimplemented unchanged -- tabulates n (valid cells) per layer x condition and per depth-group x condition, flagging anything below `low_n_threshold`.

In [ ]:
def summarize_layer_sample_sizes(df, layer_col='layer', group_col='condition', filter_col='valid',
                                  deep_layers=('L5', 'L6'), superficial_layers=('L2/3', 'L4'),
                                  low_n_threshold=10):
    """
    Tabulate n (valid cells) per layer x condition and per depth_group x
    condition, flagging anything below low_n_threshold.

    Parameters
    ----------
    df : pandas.DataFrame
        One group's table.
    layer_col, group_col, filter_col : str
    deep_layers, superficial_layers : tuple of str
    low_n_threshold : int

    Returns
    -------
    layer_counts : pandas.DataFrame
        One row per layer, one column per condition, n = valid cells.
    depth_counts : pandas.DataFrame
        Same, but for the two pooled depth groups.
    """
    filtered = df[df[filter_col]].copy()
    filtered['depth_group'] = _assign_depth_group(filtered[layer_col], deep_layers, superficial_layers)

    layers_present = [l for l in filtered[layer_col].dropna().unique()]
    layer_order = _layer_order(layers_present)
    layer_counts = filtered.groupby([layer_col, group_col]).size().unstack(fill_value=0).reindex(layer_order)

    depth_counts = filtered.groupby(['depth_group', group_col]).size().unstack(fill_value=0)
    depth_counts = depth_counts.reindex(['deep', 'superficial'])

    print("Sample sizes (valid cells) per layer x condition:")
    print(layer_counts.to_string())
    low_layer = layer_counts[layer_counts.lt(low_n_threshold).any(axis=1)]
    if len(low_layer) > 0:
        print(f"\nWARNING: layer(s) with a condition below n={low_n_threshold} "
              f"-- interpret those specific comparisons cautiously:")
        print(low_layer.to_string())

    print("\nSample sizes (valid cells) per depth group x condition:")
    print(depth_counts.to_string())
    low_depth = depth_counts[depth_counts.lt(low_n_threshold).any(axis=1)]
    if len(low_depth) > 0:
        print(f"\nWARNING: depth group(s) with a condition below n={low_n_threshold}:")
        print(low_depth.to_string())

    return layer_counts, depth_counts

## Functions -- drivers (`run_layer_analysis_for_group` / `run_layer_analysis_all_groups`)

Reimplemented unchanged -- sample-size diagnostic, then 4-layer breakdown, pooled deep/superficial comparison, and the formal interaction test, per group. No popup windows (figures `plt.close`d immediately after creation, same convention as everywhere else in this pipeline).

In [ ]:
def run_layer_analysis_for_group(df, group_name=''):
    """
    Runs the sample-size diagnostic, 4-layer breakdown, pooled
    deep/superficial comparison, and interaction test for one group.
    Both figures are annotated with a saline-vs-dcz_random_subsampled
    significance bracket, reusing compare_smi_by_layer/
    compare_smi_by_depth_group's own pairwise results directly (no
    recomputation, no duplicate printing).

    Parameters
    ----------
    df : pandas.DataFrame
        One group's table.
    group_name : str

    Returns
    -------
    result : dict with keys: layer_counts, depth_counts, layer_results,
        layer_summary, layer_fig, depth_results, depth_summary, depth_fig,
        interaction_result.
    """
    print(f"\n{'='*90}\nLayer analysis: {group_name}\n{'='*90}")

    layer_counts, depth_counts = summarize_layer_sample_sizes(df)

    layer_results, layer_summary_df = compare_smi_by_layer(df)
    layer_fig = plot_smi_by_layer(df, title=group_name, stats_by_layer=layer_results)
    plt.close(layer_fig)

    depth_results, depth_summary_df = compare_smi_by_depth_group(df)
    depth_fig = plot_smi_by_depth_group(df, title=f"{group_name} (deep vs superficial)",
                                         stats_by_depth=depth_results)
    plt.close(depth_fig)

    interaction_result = test_layer_depth_interaction(df)

    return {
        'layer_counts': layer_counts,
        'depth_counts': depth_counts,
        'layer_results': layer_results,
        'layer_summary': layer_summary_df,
        'layer_fig': layer_fig,
        'depth_results': depth_results,
        'depth_summary': depth_summary_df,
        'depth_fig': depth_fig,
        'interaction_result': interaction_result,
    }


def run_layer_analysis_all_groups(all_group_dfs, group_col='condition', filter_col='valid'):
    """
    Loops run_layer_analysis_for_group over every group, skipping
    single-condition groups (defensive check).

    Parameters
    ----------
    all_group_dfs : dict
        {group_name: df}, from load_all_group_dfs_from_random_subsample.
    group_col, filter_col : str

    Returns
    -------
    results : dict
        {group_name: run_layer_analysis_for_group(...) result}.
    """
    results = {}
    for group_name, df in all_group_dfs.items():
        conditions_present = df.loc[df[filter_col], group_col].unique()
        if len(conditions_present) < 2:
            print(f"\n{'='*90}\n{group_name}: only {len(conditions_present)} condition(s) present "
                  f"({list(conditions_present)}) -- skipping layer analysis for this group.\n{'='*90}")
            continue
        results[group_name] = run_layer_analysis_for_group(df, group_name=group_name)

    return results

## Functions -- save everything

Reimplemented unchanged -- sample-size tables, per-layer and per-depth-group summary stats (+ omnibus JSON), both figures, and the interaction regression's text summary, per group.

In [ ]:
def save_dataframe_csv(df, output_dir, filename, index=False):
    """
    Save a DataFrame to {output_dir}/{filename}, creating output_dir if
    needed. index=True for tables whose index is meaningful (e.g. layer
    names), False for tables with a plain range index.
    """
    os.makedirs(output_dir, exist_ok=True)
    save_path = os.path.join(output_dir, filename)
    df.to_csv(save_path, index=index)
    print(f"Saved -> {save_path}")
    return save_path


def save_figure_png(fig, output_dir, filename, dpi=150):
    """
    Save a matplotlib figure to {output_dir}/{filename}, creating
    output_dir if needed.
    """
    os.makedirs(output_dir, exist_ok=True)
    save_path = os.path.join(output_dir, filename)
    fig.savefig(save_path, dpi=dpi, bbox_inches='tight')
    print(f"Saved -> {save_path}")
    return save_path


def _json_safe(obj):
    """Recursively convert numpy scalar types to native Python for json.dump."""
    if isinstance(obj, dict):
        return {k: _json_safe(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [_json_safe(v) for v in obj]
    if isinstance(obj, np.floating):
        return float(obj)
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, np.bool_):
        return bool(obj)
    return obj


def save_json(data, output_dir, filename):
    """
    Save a JSON-serializable dict to {output_dir}/{filename}, creating
    output_dir if needed.
    """
    import json
    os.makedirs(output_dir, exist_ok=True)
    save_path = os.path.join(output_dir, filename)
    with open(save_path, 'w') as f:
        json.dump(_json_safe(data), f, indent=2)
    print(f"Saved -> {save_path}")
    return save_path


def _extract_omnibus_summary(results_by_category):
    """
    Pull {category: {'omnibus_stat', 'omnibus_p', 'group_medians', 'group_n'}}
    out of a {category: compare_smi_across_conditions result or None} dict,
    for JSON saving (skips categories where result was None).
    """
    summary = {}
    for cat, result in results_by_category.items():
        if result is None:
            continue
        summary[cat] = {
            'omnibus_stat': result['omnibus_stat'],
            'omnibus_p': result['omnibus_p'],
            'group_medians': result['group_medians'],
            'group_n': result['group_n'],
        }
    return summary


def save_layer_analysis_group_outputs(output_dir, group_name, result):
    """
    Save one group's layer-analysis outputs: sample-size tables, per-layer
    and per-depth-group summary stats (+ omnibus JSON), both figures, and
    the interaction regression's text summary.

    Parameters
    ----------
    output_dir : str
    group_name : str
    result : dict
        From run_layer_analysis_for_group.

    Returns
    -------
    saved_paths : dict
    """
    saved_paths = {
        'layer_sample_sizes': save_dataframe_csv(
            result['layer_counts'], output_dir, f"{group_name}_layer_sample_sizes.csv", index=True),
        'depth_sample_sizes': save_dataframe_csv(
            result['depth_counts'], output_dir, f"{group_name}_depth_sample_sizes.csv", index=True),
    }

    if len(result['layer_summary']) > 0:
        saved_paths['layer_pairwise'] = save_dataframe_csv(
            result['layer_summary'], output_dir, f"{group_name}_layer_pairwise_stats.csv")
    if len(result['depth_summary']) > 0:
        saved_paths['depth_pairwise'] = save_dataframe_csv(
            result['depth_summary'], output_dir, f"{group_name}_depth_pairwise_stats.csv")

    saved_paths['layer_omnibus'] = save_json(
        _extract_omnibus_summary(result['layer_results']), output_dir, f"{group_name}_layer_omnibus_stats.json")
    saved_paths['depth_omnibus'] = save_json(
        _extract_omnibus_summary(result['depth_results']), output_dir, f"{group_name}_depth_omnibus_stats.json")

    layer_fig = result.get('layer_fig')
    if layer_fig is not None:
        saved_paths['layer_plot'] = save_figure_png(layer_fig, output_dir, f"{group_name}_layer_comparison_plot.png")

    depth_fig = result.get('depth_fig')
    if depth_fig is not None:
        saved_paths['depth_plot'] = save_figure_png(depth_fig, output_dir, f"{group_name}_depth_comparison_plot.png")

    interaction_result = result.get('interaction_result')
    if interaction_result is not None:
        os.makedirs(output_dir, exist_ok=True)
        txt_path = os.path.join(output_dir, f"{group_name}_interaction_regression.txt")
        with open(txt_path, 'w') as f:
            f.write(str(interaction_result.summary()))
        print(f"Saved -> {txt_path}")
        saved_paths['interaction_regression'] = txt_path

    return saved_paths


def save_all_layer_analysis_outputs(output_dir, layer_analysis_results):
    """
    Loop save_layer_analysis_group_outputs over every group.

    Parameters
    ----------
    output_dir : str
    layer_analysis_results : dict
        {group_name: run_layer_analysis_for_group(...) result}.

    Returns
    -------
    saved_paths_by_group : dict
    """
    saved_paths_by_group = {}
    for group_name, result in layer_analysis_results.items():
        saved_paths_by_group[group_name] = save_layer_analysis_group_outputs(output_dir, group_name, result)
    print(f"\nSaved layer-analysis outputs for {len(saved_paths_by_group)} group(s) to {output_dir}")
    return saved_paths_by_group

## Run it -- both animals, no popup windows

Reads from `Phase4_TrialMatched_RandomSubsample_Results` (not the sequential-block notebook's `Phase4_TrialMatched_Results`), saves to its own `Phase5_LayerSpecific_RandomSubsample_Results`. Same no-blocking, dual-animal convention as everywhere else in this pipeline.

In [ ]:
ANIMAL_CONFIGS = [
    {'animal_label': 'JSY090', 'animal_dir': TEST_ANIMAL_DIR_JSY090},
    {'animal_label': 'JSY093', 'animal_dir': TEST_ANIMAL_DIR_JSY093},
]

layer_random_subsample_results_by_animal = {}
for animal_cfg in ANIMAL_CONFIGS:
    print(f"\n{'#'*90}\n{animal_cfg['animal_label']}\n{'#'*90}")
    phase4_random_subsample_dir = os.path.join(animal_cfg['animal_dir'], 'Phase4_TrialMatched_RandomSubsample_Results')
    phase5_output_dir = os.path.join(animal_cfg['animal_dir'], 'Phase5_LayerSpecific_RandomSubsample_Results')

    all_group_dfs = load_all_group_dfs_from_random_subsample(phase4_random_subsample_dir)
    layer_analysis_results = run_layer_analysis_all_groups(all_group_dfs)
    saved_paths_by_group = save_all_layer_analysis_outputs(phase5_output_dir, layer_analysis_results)

    layer_random_subsample_results_by_animal[animal_cfg['animal_label']] = {
        'all_group_dfs': all_group_dfs,
        'layer_analysis_results': layer_analysis_results,
        'saved_paths_by_group': saved_paths_by_group,
    }